# 🐍 Master Guide: 01: File Handling, Binary Streams & OS File System

> **A Complete Foundations-to-Advanced Reference Guide**  
> Covers: Data Persistency Concepts, The 8 File Modes Matrix, Cursor Navigation (`seek` and `tell`), Binary File Copying, and OS Directory Management (`os.walk`).
---

## 1. Core Purpose: Data Persistency & Streams

> **Definition**: When a Python program terminates, all data stored in RAM variables is destroyed. **File Handling** achieves **Data Persistency** by writing structured information to non-volatile secondary storage (SSD/HDD).
>
> **What is a File / Stream?**
> A **File** is a named location on secondary storage that stores a sequence of bytes.
> In Python, files are managed through **Stream Objects** created by the built-in `open()` function:
>
> $$\text{file\_object} = \mathbf{open}(\text{filename}, \; \text{mode}='r', \; [\text{buffering}], \; [\text{encoding}='utf-8'])$$

## 2. The 8 File Access Modes Matrix

| Mode | Meaning | Creates New File? | Truncates (Overwrites)? | Cursor Position |
|---|---|---|---|---|
| `'r'` | Read Only | **No** (Raises `FileNotFoundError`) | No | Beginning of file (0) |
| `'w'` | Write Only | **Yes** | **Yes** (Wipes existing content) | Beginning of file (0) |
| `'a'` | Append Only | **Yes** | No (Preserves existing data) | End of file |
| `'r+'`| Read and Write | **No** (Must exist) | No | Beginning of file (0) |
| `'w+'`| Write and Read | **Yes** | **Yes** (Wipes existing content) | Beginning of file (0) |
| `'a+'`| Append and Read | **Yes** | No | End of file |
| `'rb'` / `'wb'` | Binary Read / Write | Same as `'r'` / `'w'` | Same | Used for images, audio, PDFs, pickles. |
| `'x'` | Exclusive Creation | **Yes** | No | Raises `FileExistsError` if file exists. |

## 3. Cursor Navigation: `tell()` and `seek()`

- **`file.tell()`**: Returns the current byte offset position of the read/write pointer.
- **`file.seek(offset, whence)`**: Moves the pointer to a new position:
  - `whence = 0`: Relative to the **beginning of the file** (Default).
  - `whence = 1`: Relative to the **current position** (Binary mode only).
  - `whence = 2`: Relative to the **end of the file** (Binary mode only).

In [ ]:
# File Cursor Navigation Demonstration with io.BytesIO
import io

# Using in-memory stream buffer to avoid cluttering disk
buffer = io.BytesIO()
buffer.write(b"ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")

print("Initial cursor position via tell():", buffer.tell())

# Reposition pointer to byte offset 10
buffer.seek(10, 0)  # Whence 0: From beginning
print("Cursor after seek(10, 0)        :", buffer.tell())
read_chunk = buffer.read(5)
print("Read 5 bytes ('K' to 'O')         :", read_chunk.decode('ascii'))
print("Cursor after reading 5 bytes     :", buffer.tell())

# Seek relative to end of file
buffer.seek(-10, 2)  # Whence 2: 10 bytes before end
tail_chunk = buffer.read()
print("Tail 10 bytes read                :", tail_chunk.decode('ascii'))

## 4. Binary File Copying Protocol (Buffered Chunking)

Reading an entire 2GB video or dataset into RAM at once with `.read()` causes memory exhaustion crashes.
Production code uses **chunked buffer streaming** (typically 4KB to 64KB blocks) to copy files with constant $O(1)$ RAM usage.

In [ ]:
# Chunked Binary Stream Copying Protocol
import io

def copy_stream_chunked(src_stream, dest_stream, chunk_size=1024):
    bytes_copied = 0
    while True:
        block = src_stream.read(chunk_size)
        if not block:
            break  # End of stream (EOF)
        dest_stream.write(block)
        bytes_copied += len(block)
    return bytes_copied

# Simulating binary image copying
mock_image = io.BytesIO(b"\x89PNG\r\n\x1a\n" + (b"\x00\xFF" * 2048))
destination = io.BytesIO()

total = copy_stream_chunked(mock_image, destination, chunk_size=512)
print(f"Binary Stream Copy Complete: {total:,} bytes copied smoothly in 512-byte blocks.")

## 5. OS File System Management (`os` & `os.path`)

- `os.mkdir(path)`: Creates a single directory.
- `os.makedirs(path, exist_ok=True)`: Creates nested directory hierarchies recursively.
- `os.walk(path)`: Yields 3-tuples `(dirpath, dirnames, filenames)` for recursive directory tree traversal.
- `os.remove(path)`: Deletes a file.
- `os.rename(old, new)`: Renames or moves a file.

In [ ]:
# OS Directory Hierarchy & Tree Traversal Suite
import os
import tempfile

with tempfile.TemporaryDirectory() as temp_dir:
    # 1. Create nested hierarchy
    proj_path = os.path.join(temp_dir, "project", "src", "core")
    os.makedirs(proj_path, exist_ok=True)
    
    # Create mock files
    with open(os.path.join(proj_path, "app.py"), "w") as f:
        f.write("print('Hello')")
    with open(os.path.join(temp_dir, "project", "README.md"), "w") as f:
        f.write("# Project Docs")

    print("--- Recursive Directory Tree Traversal (os.walk) ---")
    for root, dirs, files in os.walk(temp_dir):
        level = root.replace(temp_dir, '').count(os.sep)
        indent = ' ' * 4 * level
        print(f"{indent}{os.path.basename(root)}/")
        for fname in files:
            print(f"{indent}    {fname}")

## 6. Summary & Quick Revision Cheat Sheet

| Concept | Method / Tool | Operational Rule |
|---|---|---|
| **Read Mode** | `'r'` | Default mode. Fails if file is missing. |
| **Write Mode** | `'w'` | Creates file or completely truncates existing contents. |
| **Append Mode** | `'a'` | Preserves existing content; writes strictly to end. |
| **Binary Mode** | `'rb'`, `'wb'` | Reads/writes raw bytes without Unicode decoding. |
| **Cursor Query** | `f.tell()` | Returns current offset in bytes. |
| **Cursor Move** | `f.seek(offset, 0)` | Moves pointer relative to beginning of file. |
| **Tree Walk** | `os.walk(path)` | Recursively explores all directories and sub-files. |